In [1]:
"""
Data Collection Script (Raw)
Fetch data from multiple sources
"""

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import feedparser

# ----------------------------- Helper -----------------------------
def fetch_url(url, retries=3, delay=2):
    for attempt in range(retries):
        try:
            response = requests.get(url, timeout=5)
            response.encoding = "utf-8"
            if response.status_code == 200:
                return response.text
        except Exception as e:
            print(f"Attempt {attempt + 1} failed for {url}: {e}")
            time.sleep(delay)
    return None

In [2]:
# ----------------------------- Books to Scrape -----------------------------
print("Fetching Books to Scrape data")
books_data = []
base_url = "http://books.toscrape.com/catalogue/page-{}.html"
book_base_url = "http://books.toscrape.com/catalogue/"

for page in range(1, 51):
    print(f"Books page {page}")
    html = fetch_url(base_url.format(page))
    if not html:
        continue
    soup = BeautifulSoup(html, 'html.parser')
    books = soup.find_all('article', class_='product_pod')

    for book in books:
        title = book.h3.a['title']
        price = book.find('p', class_='price_color').text
        rating = book.p['class'][1]  # raw text rating (e.g., "Three")
        availability = book.find('p', class_='instock availability').text.strip()

        # follow link to book detail page to get category
        book_url = book_base_url + book.h3.a['href']
        book_html = fetch_url(book_url)
        category = "Unknown"
        if book_html:
            detail_soup = BeautifulSoup(book_html, 'html.parser')
            breadcrumb = detail_soup.find('ul', class_='breadcrumb')
            if breadcrumb:
                li_tags = breadcrumb.find_all('li')
                if len(li_tags) > 2:
                    category = li_tags[2].text.strip()

        books_data.append({
            'source': 'books.toscrape',
            'title': title,
            'price': price,
            'rating': rating,
            'category': category,
            'availability': availability
        })
        time.sleep(random.uniform(0.5, 1.5))

    time.sleep(random.uniform(1, 2))

books_df = pd.DataFrame(books_data)

Fetching Books to Scrape data
Books page 1
Books page 2
Books page 3
Books page 4
Books page 5
Books page 6
Books page 7
Books page 8
Books page 9
Books page 10
Books page 11
Books page 12
Books page 13
Books page 14
Books page 15
Books page 16
Books page 17
Books page 18
Books page 19
Books page 20
Books page 21
Books page 22
Books page 23
Books page 24
Books page 25
Books page 26
Books page 27
Books page 28
Books page 29
Books page 30
Books page 31
Books page 32
Books page 33
Books page 34
Books page 35
Books page 36
Books page 37
Books page 38
Books page 39
Books page 40
Books page 41
Books page 42
Books page 43
Books page 44
Books page 45
Books page 46
Books page 47
Books page 48
Books page 49
Books page 50


In [3]:
# ----------------------------- Fake Store API -----------------------------
print("Fetching Fake Store API data")
fake_api_url = "https://fakestoreapi.com/products"
try:
    response = requests.get(fake_api_url, timeout=5)
    fake_data = response.json() if response.status_code == 200 else []
except Exception as e:
    print("Error fetching Fake Store API:", e)
    fake_data = []

fake_store_data = []
for item in fake_data:
    fake_store_data.append({
        'source': 'fakestoreapi',
        'title': item.get('title'),
        'price': item.get('price'),
        'rating': item.get('rating', {}).get('rate'),
        'category': item.get('category'),
        'availability': "In Stock"
    })

fake_store_df = pd.DataFrame(fake_store_data)


Fetching Fake Store API data


In [4]:
# ----------------------------- NYT Books RSS -----------------------------
print("Fetching NYT Books RSS feed")
rss_url = "https://rss.nytimes.com/services/xml/rss/nyt/Books.xml"
feed = feedparser.parse(rss_url)

nyt_data = []
for entry in feed.entries[:30]:
    nyt_data.append({
        'source': 'nyt_books_rss',
        'title': entry.title,
        'price': None,
        'rating': None,
        'category': entry.tags[0]['term'] if entry.get('tags') else 'Books',
        'availability': None
    })

nyt_df = pd.DataFrame(nyt_data)

Fetching NYT Books RSS feed


In [5]:
# ----------------------------- Combine and Save RAW -----------------------------
combined_df = pd.concat([books_df, fake_store_df, nyt_df], ignore_index=True)

combined_df.to_csv("raw_book_data.csv", index=False, encoding="utf-8-sig")
print("Data Collected Successfully.")

Data Collected Successfully.
